# H5 Functionalities
Lets try to extract the data from H5 files. 

In [ ]:
from ase import Atoms
from ase.build import molecule
from ase.units import Ha
from ase_dirac import DIRAC
import h5py
import numpy as np
from typing import Optional, Union
import matplotlib.pyplot as plt

In [ ]:
import ase


molecule_label = "H2"
ase_molecule = molecule(molecule_label)

inp_string = """
        **DIRAC
        .TITLE
        Relativistic SCF on H2 
        .WAVE F
        **HAMILTONIAN
        .LVCORR
        **WAVE FUNCTIONS
        .SCF
        *SCF
        .PRINT
        1
        **MOLECULE
        *BASIS
        .DEFAULT
        dyall.3zp
    """

ase_molecule.calc = DIRAC.calculator_from_string(inp_string)
ase_molecule.calc.prefix = 'H2'

print(
    f"The Hartree-Fock energy of {molecule_label} is ",
    ase_molecule.get_potential_energy() / Ha,
    " Hartree.",
)

In [ ]:
def show(name, obj):
    if isinstance(obj, h5py.Group):
        print("Group:", name)
    elif isinstance(obj, h5py.Dataset):
        print("Dataset:", name, obj.shape)


f = h5py.File(f"h2_symm.h5", "r")
# f = h5py.File(f"H2.h5", "r")
f.visititems(show)
data_L = f["input/aobasis/1/"]
data_S = f["input/aobasis/2/"]
one_electron_ints= f["result/operators/ao_matrices"]
eigval = f["result/wavefunctions/scf/mobasis/eigenvalues"]
eigval[()]

## Large component
First we will take a look at the large component basis:

In [ ]:
for key in data_L.keys():
    print(key, data_L[key][()])

Lets analyze each keyword-value pair: 

Keywords `centers`, `contractions`, and `exponents` are self-explanatory. Now,`n_ao` corresponds with the total number of Large component basis:

In [ ]:
!grep 'Atoms and basis sets' H2.out -A 16

The `orbmom` list contains the angular momentum definition of the basis: 

In [ ]:
basisdict = {
    "s": 0,
    "p": 0,
    "d": 0,
    "f": 0,
    "g": 0,
    "h": 0,
}

amlist = ["s", "p", "d", "f", "g", "h"]

for val in data_L["orbmom"][()]:
    basisdict[amlist[val - 1]] += 1 / 2

basisstr = ""
for key in basisdict.keys():
    if basisdict[key] != 0:
        basisstr += f"{int(basisdict[key])}{key}"

basisstr_L = basisstr

print(f"The contracted LC basis function is: {basisstr}.")

total_basis = 0


def cartesian_projections(l: int) -> int:
    return (l + 1) * (l + 2) // 2


for key in basisdict.keys():
    if basisdict[key] != 0:
        total_basis += int(basisdict[key]) * cartesian_projections(amlist.index(key))

print(
    f"The LC basis functions are {total_basis} per atom, so {total_basis*2} in total as expected in the previous grep."
)

And for the small component:

In [ ]:
basisdict = {
    "s": 0,
    "p": 0,
    "d": 0,
    "f": 0,
    "g": 0,
    "h": 0,
}

amlist = ["s", "p", "d", "f", "g", "h"]

for val in data_S["orbmom"][()]:
    basisdict[amlist[val - 1]] += 1 / 2

basisstr = ""
for key in basisdict.keys():
    if basisdict[key] != 0:
        basisstr += f"{int(basisdict[key])}{key}"

basisstr_S = basisstr

print(f"The contracted SC basis function is: {basisstr}.")

total_basis_S = 0

for key in basisdict.keys():
    if basisdict[key] != 0:
        total_basis_S += int(basisdict[key]) * cartesian_projections(amlist.index(key))

print(
    f"The SC basis functions are {total_basis_S} per atom, so {total_basis_S*2} in total as expected in the previous grep."
)

print(f"The total number of LC + SC basis are {2*(total_basis_S + total_basis)}.")

## Matrix elements
Let's take a look now at the one electron matrix elements in the `.h5` file:

In [ ]:
one_electron_ints.keys()

for key in one_electron_ints.keys():
    print(key, one_electron_ints[key][()].shape, one_electron_ints[key][()].dtype)

The dimension of the basis (LC + SC) is: 

In [ ]:
one_electron_ints["aobasis_dim"][0]

Now, `BETAMAT FFFT`, `MOLFIELDTFFT`, and `OVERLAP TFFT` are one-dimensional arrays. Therefore, this means that we have to manipulate them to return to the typical two-dimensional form. 


In [ ]:
def retriangularize_and_conjugate(array, n_basis):
    M = np.zeros((n_basis, n_basis), dtype=array.dtype)
    k = 0
    for j in range(n_basis):
        for i in range(j + 1):
            M[i, j] = array[k]
            M[j, i] = np.conjugate(array[k])
            k += 1
    return M


def plot_packed_heatmap(
    array,
    n,
    LC_size: Optional[Union[int, None]] = None,
    title="Packed matrix heatmap",
    cmap="viridis",
    only_LC: bool = False,
):
    M = retriangularize_and_conjugate(array, n)
    Mplot = M.real if np.iscomplexobj(M) else M

    fig, ax = plt.subplots(figsize=(8, 7))
    if only_LC: 
        ax.set_xlim(0,LC_size)
        ax.set_ylim(0,LC_size)
    
    im = ax.imshow(Mplot, cmap=cmap, origin="upper", aspect="auto")
    fig.colorbar(im, ax=ax, label="Matrix element")

    ax.set_xlabel("AO index")
    ax.set_ylabel("AO index")
    ax.set_title(title)

    if LC_size is not None:
        ax.axvline(x=LC_size - 0.5, color="black", linewidth=1.5)
        ax.axhline(y=LC_size - 0.5, color="black", linewidth=1.5)


    plt.tight_layout()
    plt.show()

    return M


n = int(one_electron_ints["aobasis_dim"][0])
one_electron_ints["OVERLAP TFFT"][:]
array = one_electron_ints["OVERLAP TFFT"][:]
M = plot_packed_heatmap(
    array,
    n,
    LC_size=data_L["n_ao"][0],
    title=f"Overlap [LC: {basisstr_L} | SC: {basisstr_S}]",
    only_LC=True,
)

one_electron_ints["OVERLAP TFFT"][:]
array = one_electron_ints["OVERLAP TFFT"][:]
M = plot_packed_heatmap(
    array,
    n,
    LC_size=data_L["n_ao"][0],
    title=f"Overlap [LC: {basisstr_L} | SC: {basisstr_S}]",
)

In [ ]:
n = int(one_electron_ints['aobasis_dim'][0])
array = one_electron_ints['MOLFIELDTFFT'][:]
M = plot_packed_heatmap(array, n, LC_size=data_L['n_ao'][0], title=f"Molfield [LC: {basisstr_L} | SC: {basisstr_S}]")

In [ ]:
n = int(one_electron_ints['aobasis_dim'][0])
array = one_electron_ints['BETAMAT FFFT'][:]
M = plot_packed_heatmap(array, n, LC_size=data_L['n_ao'][0], title=f"Betamat [LC: {basisstr_L} | SC: {basisstr_S}]")